# 02. Graph Traversal: DFS and BFS

A traversal visits every vertex reachable from a start. The new thing compared to trees: graphs can have **cycles**, so we must remember **visited** vertices.

**Learning order:** DFS → BFS → VISITED RULE → CONNECTED COMPONENTS → PATH EXISTS → ALL PATHS → CLONE GRAPH → DFS VS BFS → INTERVIEW THINKING

## 1. DFS: go deep first

Follow one path as far as possible, then **backtrack**. Uses recursion or an explicit stack. Real life: exploring a maze by always taking the first unexplored corridor.

In [1]:
from collections import deque, defaultdict

graph = {0: [1, 2], 1: [3], 2: [3], 3: []}

def dfs_recursive(graph, start):
    visited, order = set(), []
    def dfs(node):
        visited.add(node)
        order.append(node)
        for nb in graph[node]:
            if nb not in visited:
                dfs(nb)
    dfs(start)
    return order

def dfs_iterative(graph, start):
    visited, order, stack = set(), [], [start]
    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        order.append(node)
        for nb in reversed(graph[node]):             # reversed to match the recursive order
            if nb not in visited:
                stack.append(nb)
    return order

print(dfs_recursive(graph, 0), dfs_iterative(graph, 0))   # [0, 1, 3, 2] [0, 1, 3, 2]

[0, 1, 3, 2] [0, 1, 3, 2]


## 2. BFS: level by level

Visit all neighbours first, then their neighbours. Uses a **queue**. Gives the **fewest edges** from the start to every vertex (shortest path in an unweighted graph).

In [2]:
def bfs(graph, start):
    visited, order = {start}, []
    q = deque([start])
    while q:
        node = q.popleft()
        order.append(node)
        for nb in graph[node]:
            if nb not in visited:
                visited.add(nb)                      # mark when ENQUEUING
                q.append(nb)
    return order

print(bfs(graph, 0))   # [0, 1, 2, 3]

[0, 1, 2, 3]


## 3. The visited rule

Without `visited`, a cycle makes the traversal run **forever**. Mark a node visited:
- BFS: **when you add it to the queue** (so it is never queued twice).
- DFS: when you **enter** it.

**Complexity (both):** O(V + E) time (each vertex once, each edge looked at once or twice), O(V) space.

## 4. Number of connected components

Loop over all vertices. Each time you find an **unvisited** vertex, start a traversal (it marks the whole component) and count `+1`.

Real life: counting separate friend groups, or isolated networks.

In [3]:
def count_components(n, edges):
    adj = defaultdict(list)
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    visited, count = set(), 0
    def dfs(node):
        visited.add(node)
        for nb in adj[node]:
            if nb not in visited:
                dfs(nb)
    for node in range(n):
        if node not in visited:
            dfs(node)
            count += 1
    return count

print(count_components(5, [(0, 1), (1, 2), (3, 4)]))   # 2

2


## 5. Find if a path exists

Traverse from `source`; if `destination` is visited, return `True`.

In [4]:
def valid_path(n, edges, source, destination):
    adj = defaultdict(list)
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    seen, stack = {source}, [source]
    while stack:
        node = stack.pop()
        if node == destination:
            return True
        for nb in adj[node]:
            if nb not in seen:
                seen.add(nb)
                stack.append(nb)
    return False

print(valid_path(3, [[0, 1], [1, 2], [2, 0]], 0, 2))                        # True
print(valid_path(6, [[0, 1], [0, 2], [3, 5], [5, 4], [4, 3]], 0, 5))        # False

True
False


## 6. All paths from source to target (DAG)

In a DAG there are no cycles, so no `visited` set is needed. Use **backtracking** (path list, append, recurse, pop).

In [5]:
def all_paths(graph):                                 # graph: list of lists, target is the last node
    target, res, path = len(graph) - 1, [], [0]
    def dfs(node):
        if node == target:
            res.append(path[:])
            return
        for nb in graph[node]:
            path.append(nb)
            dfs(nb)
            path.pop()
    dfs(0)
    return res

print(all_paths([[1, 2], [3], [3], []]))   # [[0, 1, 3], [0, 2, 3]]

[[0, 1, 3], [0, 2, 3]]


## 7. Clone Graph (deep copy)

Map **old node → new node** with a dictionary. When you meet a node already in the map, **reuse its copy** (this also breaks cycles).

In [6]:
class Node:
    def __init__(self, val=0):
        self.val = val
        self.neighbors = []

def clone_graph(node):
    if not node:
        return None
    copies = {node: Node(node.val)}
    q = deque([node])
    while q:
        cur = q.popleft()
        for nb in cur.neighbors:
            if nb not in copies:
                copies[nb] = Node(nb.val)
                q.append(nb)
            copies[cur].neighbors.append(copies[nb])
    return copies[node]

a, b, c, d = Node(1), Node(2), Node(3), Node(4)
a.neighbors, b.neighbors, c.neighbors, d.neighbors = [b, d], [a, c], [b, d], [a, c]
copy = clone_graph(a)
print(copy is not a, [n.val for n in copy.neighbors], copy.neighbors[0] is not b)   # True [2, 4] True

True [2, 4] True


## 8. DFS vs BFS: which one?

| Need | Use |
|---|---|
| shortest path / fewest steps in an **unweighted** graph | **BFS** |
| explore everything, components, cycle detection | DFS or BFS |
| all paths, backtracking, topological sort | **DFS** |
| very deep graph (recursion limit) | iterative DFS or BFS |
| level-by-level processing | BFS |

## 9. Interview thinking

**Say:** "I'll traverse with BFS/DFS from each unvisited node, tracking visited in a set. O(V + E) time, O(V) space."

**Common mistakes:** missing `visited` (infinite loop), marking visited on pop in BFS (duplicates), forgetting disconnected components (loop over all nodes), recursion limit on a long path (use iterative), using BFS distance for weighted graphs.

## Quick Revision

- DFS = stack/recursion, BFS = queue. Both O(V + E).
- Visited set always (except DAG all-paths backtracking).
- Components: loop over all nodes, count traversals started.
- BFS = shortest path in unweighted graphs.
- Clone: map old → new.